In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('../data/data_spam_KKP.csv')
df

In [ ]:
# Import libraries
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("../models/v3-tuned")
model = AutoModelForSequenceClassification.from_pretrained("../models/v3-tuned")

In [ ]:
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

# ====== CONFIG ======
LABEL_NAMES = ["HAM", "SPAM"]  # urutan sesuai encoding model
BATCH_SIZE = 16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ====== LOAD DATA EXCEL ======
df_test = pd.read_excel("../data/spam_v5_train.xlsx")  

# Ambil baris 12003 sampai 40000
df_test = df_test.iloc[12003:40000].reset_index(drop=True)

texts = df_test["text"].tolist()

# ====== TOKENISASI ======
encodings = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

dataset = torch.utils.data.TensorDataset(
    encodings["input_ids"],
    encodings["attention_mask"]
)

dataloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE)

# ====== PREDIKSI ======
model = model.to(device)
model.eval()
predictions_all = []
confidences_all = []

with torch.no_grad():
    for batch in dataloader:
        input_ids, attention_mask = batch
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        
        probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()
        preds = probs.argmax(axis=-1)
        
        predictions_all.extend(preds)
        confidences_all.extend(probs.max(axis=-1))

# ====== SIMPAN HASIL ======
pred_labels_str = [LABEL_NAMES[p] for p in predictions_all]
df_test["predicted_label"] = pred_labels_str
df_test["confidence"] = confidences_all

df_test.to_excel("../data/hasil_prediksi_spam.xlsx", index=False)
print("\nHasil prediksi disimpan ke '../data/hasil_prediksi_spam.xlsx'")
